In [ ]:
import numpy as np
import pandas as pd

import optuna

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.callbacks import LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.optimizers.schedules import ExponentialDecay

from sklearn.metrics import mean_absolute_error as mae
from sklearn.preprocessing import RobustScaler, normalize
from sklearn.model_selection import train_test_split, GroupKFold, KFold

from IPython.display import display

In [ ]:
DEBUG = False

train = pd.read_csv('../input/ventilator-pressure-prediction/train.csv')
test = pd.read_csv('../input/ventilator-pressure-prediction/test.csv')
submission = pd.read_csv('../input/ventilator-pressure-prediction/sample_submission.csv')

if DEBUG:
    train = train[:80*1000]

In [ ]:
# def add_features(df):
#     # rewritten calculation of lag features from this notebook: https://www.kaggle.com/patrick0302/add-lag-u-in-as-new-feat
#     # some of ideas from this notebook: https://www.kaggle.com/mst8823/google-brain-lightgbm-baseline
#     df['last_value_u_in'] = df.groupby('breath_id')['u_in'].transform('last')
#     df['u_in_lag1'] = df.groupby('breath_id')['u_in'].shift(1)
#     df['u_out_lag1'] = df.groupby('breath_id')['u_out'].shift(1)
#     df['u_in_lag_back1'] = df.groupby('breath_id')['u_in'].shift(-1)
#     df['u_out_lag_back1'] = df.groupby('breath_id')['u_out'].shift(-1)
#     df['u_in_lag2'] = df.groupby('breath_id')['u_in'].shift(2)
#     df['u_out_lag2'] = df.groupby('breath_id')['u_out'].shift(2)
#     df['u_in_lag_back2'] = df.groupby('breath_id')['u_in'].shift(-2)
#     df['u_out_lag_back2'] = df.groupby('breath_id')['u_out'].shift(-2)
#     df['u_in_lag3'] = df.groupby('breath_id')['u_in'].shift(3)
#     df['u_out_lag3'] = df.groupby('breath_id')['u_out'].shift(3)
#     df['u_in_lag_back3'] = df.groupby('breath_id')['u_in'].shift(-3)
#     df['u_out_lag_back3'] = df.groupby('breath_id')['u_out'].shift(-3)
#     df = df.fillna(0)


#     df['R__C'] = df["R"].astype(str) + '__' + df["C"].astype(str)

#     # max value of u_in and u_out for each breath
#     df['breath_id__u_in__max'] = df.groupby(['breath_id'])['u_in'].transform('max')
#     df['breath_id__u_out__max'] = df.groupby(['breath_id'])['u_out'].transform('max')

#     # difference between consequitive values
#     df['u_in_diff1'] = df['u_in'] - df['u_in_lag1']
#     df['u_out_diff1'] = df['u_out'] - df['u_out_lag1']
#     df['u_in_diff2'] = df['u_in'] - df['u_in_lag2']
#     df['u_out_diff2'] = df['u_out'] - df['u_out_lag2']
#     # from here: https://www.kaggle.com/yasufuminakama/ventilator-pressure-lstm-starter
#     df.loc[df['time_step'] == 0, 'u_in_diff'] = 0
#     df.loc[df['time_step'] == 0, 'u_out_diff'] = 0

#     # difference between the current value of u_in and the max value within the breath
#     df['breath_id__u_in__diffmax'] = df.groupby(['breath_id'])['u_in'].transform('max') - df['u_in']
#     df['breath_id__u_in__diffmean'] = df.groupby(['breath_id'])['u_in'].transform('mean') - df['u_in']

#     # OHE
#     df = df.merge(pd.get_dummies(df['R'], prefix='R'), left_index=True, right_index=True).drop(['R'], axis=1)
#     df = df.merge(pd.get_dummies(df['C'], prefix='C'), left_index=True, right_index=True).drop(['C'], axis=1)
#     df = df.merge(pd.get_dummies(df['R__C'], prefix='R__C'), left_index=True, right_index=True).drop(['R__C'], axis=1)

#     # https://www.kaggle.com/c/ventilator-pressure-prediction/discussion/273974
#     df['u_in_cumsum'] = df.groupby(['breath_id'])['u_in'].cumsum()
#     df['time_step_cumsum'] = df.groupby(['breath_id'])['time_step'].cumsum()
#     return df

In [ ]:
# def add_feats(train):
#     # # rewritten calculation of lag features from this notebook: https://www.kaggle.com/patrick0302/add-lag-u-in-as-new-feat
# # # some of ideas from this notebook: https://www.kaggle.com/mst8823/google-brain-lightgbm-baseline
#     # train[["15_out_mean"]] = train.groupby('breath_id')['u_out'].rolling(window=15,min_periods=1).agg({"15_out_mean":"mean"}).reset_index(level=0,drop=True)
#     train['last_value_u_in'] = train.groupby('breath_id')['u_in'].transform('last')
#     train['u_in_lag1'] = train.groupby('breath_id')['u_in'].shift(1)
#     train['u_out_lag1'] = train.groupby('breath_id')['u_out'].shift(1)
#     train['u_in_lag_back1'] = train.groupby('breath_id')['u_in'].shift(-1)
#     train['u_out_lag_back1'] = train.groupby('breath_id')['u_out'].shift(-1)
#     train['u_in_lag2'] = train.groupby('breath_id')['u_in'].shift(2)
#     train['u_out_lag2'] = train.groupby('breath_id')['u_out'].shift(2)
#     train['u_in_lag3'] = train.groupby('breath_id')['u_in'].shift(3)
#     train['u_out_lag3'] = train.groupby('breath_id')['u_out'].shift(3)
#     train['u_in_lag_back2'] = train.groupby('breath_id')['u_in'].shift(-2)
#     train['u_out_lag_back2'] = train.groupby('breath_id')['u_out'].shift(-2)
#     train['u_in_lag_back3'] = train.groupby('breath_id')['u_in'].shift(-3)
#     train['u_out_lag_back3'] = train.groupby('breath_id')['u_out'].shift(-3)
#     train['u_in_lag_back10'] = train.groupby('breath_id')['u_in'].shift(-10)
#     train['u_out_lag_back10'] = train.groupby('breath_id')['u_out'].shift(-10)

#     train['u_in_first'] = train.groupby('breath_id')['u_in'].first()
#     train['u_out_first'] = train.groupby('breath_id')['u_out'].first()

#     ## time since last step
#     train['time_step_diff'] = train.groupby('breath_id')['time_step'].diff().fillna(0)
#     ### rolling window ts feats
#     train['ewm_u_in_mean'] = train.groupby('breath_id')['u_in'].ewm(halflife=9).mean().reset_index(level=0,drop=True)
#     train['ewm_u_in_std'] = train.groupby('breath_id')['u_in'].ewm(halflife=10).std().reset_index(level=0,drop=True) ## could add covar?
#     train['ewm_u_in_corr'] = train.groupby('breath_id')['u_in'].ewm(halflife=15).corr().reset_index(level=0,drop=True) # self umin corr
#     # train['ewm_u_in_corr'] = train.groupby('breath_id')['u_in'].ewm(halflife=6).corr(train.groupby('breath_id')["u_out"]).reset_index(level=0,drop=True) # corr with u_out # error
#     ## rolling window of 15 periods
#     train[["15_in_sum","15_in_min","15_in_max","15_in_mean","15_out_std"]] = train.groupby('breath_id')['u_in'].rolling(window=15,min_periods=1).agg({"15_in_sum":"sum","15_in_min":"min","15_in_max":"max","15_in_mean":"mean","15_in_std":"std"}).reset_index(level=0,drop=True)
# #     train[["45_in_sum","45_in_min","45_in_max","45_in_mean","45_out_std"]] = train.groupby('breath_id')['u_in'].rolling(window=45,min_periods=1).agg({"45_in_sum":"sum","45_in_min":"min","45_in_max":"max","45_in_mean":"mean","45_in_std":"std"}).reset_index(level=0,drop=True)
#     train[["45_in_sum","45_in_min","45_in_max","45_in_mean","45_out_std"]] = train.groupby('breath_id')['u_in'].rolling(window=45,min_periods=1).agg({"45_in_sum":"sum","45_in_min":"min","45_in_max":"max","45_in_mean":"mean","45_in_std":"std"}).reset_index(level=0,drop=True)

#     train[["15_out_mean"]] = train.groupby('breath_id')['u_out'].rolling(window=15,min_periods=1).agg({"15_out_mean":"mean"}).reset_index(level=0,drop=True)

#     print(train.shape[0])
#     display(train)
#     train = train.fillna(0) # ORIG

#     # max, min, mean value of u_in and u_out for each breath
#     train['breath_id__u_in__max'] = train.groupby(['breath_id'])['u_in'].transform('max')
#     train['breath_id__u_out__max'] = train.groupby(['breath_id'])['u_out'].transform('max')

#     train['breath_id__u_out__mean'] =train.groupby(['breath_id'])['u_out'].mean()
#     train['breath_id__u_in__mean'] =train.groupby(['breath_id'])['u_in'].mean()

#     train['breath_id__u_in__min'] = train.groupby(['breath_id'])['u_in'].transform('min')
#     train['breath_id__u_out__min'] = train.groupby(['breath_id'])['u_out'].transform('min')

#     train['R_div_C'] = train["R"].div(train["C"])

#     # difference between consequitive values
#     train['R__C'] = train["R"].astype(str) + '__' + train["C"].astype(str)
#     train['u_in_diff1'] = train['u_in'] - train['u_in_lag1']
#     train['u_out_diff1'] = train['u_out'] - train['u_out_lag1']
#     train['u_in_diff2'] = train['u_in'] - train['u_in_lag2']
#     train['u_out_diff2'] = train['u_out'] - train['u_out_lag2']
#     train['u_in_diff3'] = train['u_in'] - train['u_in_lag3']
#     train['u_out_diff3'] = train['u_out'] - train['u_out_lag3']
#     ## diff between last 2 steps
#     train['u_in_diff_1_2'] = train['u_in_lag1'] - train['u_in_lag2']
#     train['u_out_diff_1_2'] = train['u_out_lag1'] - train['u_out_lag2']
#     train['u_in_lagback_diff_1_2'] = train['u_in_lag_back1'] - train['u_in_lag_back2']
#     train['u_out_lagback_diff_1_2'] = train['u_out_lag_back1'] - train['u_out_lag_back2']

#     train['u_in_lagback_diff1'] = train['u_in'] - train['u_in_lag_back1']
#     train['u_out_lagback_diff1'] = train['u_out'] - train['u_out_lag_back1']
#     train['u_in_lagback_diff2'] = train['u_in'] - train['u_in_lag_back2']
#     train['u_out_lagback_diff2'] = train['u_out'] - train['u_out_lag_back2']

#     # from here: https://www.kaggle.com/yasufuminakama/ventilator-pressure-lstm-starter
#     train.loc[train['time_step'] == 0, 'u_in_diff'] = 0
#     train.loc[train['time_step'] == 0, 'u_out_diff'] = 0

#     # difference between the current value of u_in and the max value within the breath
#     train['breath_id__u_in__diffmax'] = train.groupby(['breath_id'])['u_in'].transform('max') - train['u_in']
#     train['breath_id__u_in__diffmean'] = train.groupby(['breath_id'])['u_in'].transform('mean') - train['u_in']

#     print("before OHE")
#     display(train)

#     # OHE
#     train = train.merge(pd.get_dummies(train['R'], prefix='R'), left_index=True, right_index=True).drop(['R'], axis=1)
#     train = train.merge(pd.get_dummies(train['C'], prefix='C'), left_index=True, right_index=True).drop(['C'], axis=1)
#     train = train.merge(pd.get_dummies(train['R__C'], prefix='R__C'), left_index=True, right_index=True).drop(['R__C'], axis=1)

#     # https://www.kaggle.com/c/ventilator-pressure-prediction/discussion/273974
#     train['u_in_cumsum'] = train.groupby(['breath_id'])['u_in'].cumsum()
#     train['time_step_cumsum'] = train.groupby(['breath_id'])['time_step'].cumsum()

#     # feature by u in or out (ideally - make 2 sep columns for each state) # dan
#     train['u_in_partition_out_sum'] = train.groupby(['breath_id',"u_out"])['u_in'].transform("sum")

#     train = train.fillna(0) # add for consistency with how test is done - dan

#     return train

In [ ]:
def add_features(df):
    df['area'] = df['time_step'] * df['u_in']
    df['area'] = df.groupby('breath_id')['area'].cumsum()
    
    df['u_in_cumsum'] = (df['u_in']).groupby(df['breath_id']).cumsum()
    
    df['u_in_lag1'] = df.groupby('breath_id')['u_in'].shift(1)
    df['u_out_lag1'] = df.groupby('breath_id')['u_out'].shift(1)
    df['u_in_lag_back1'] = df.groupby('breath_id')['u_in'].shift(-1)
    df['u_out_lag_back1'] = df.groupby('breath_id')['u_out'].shift(-1)
    df['u_in_lag2'] = df.groupby('breath_id')['u_in'].shift(2)
    df['u_out_lag2'] = df.groupby('breath_id')['u_out'].shift(2)
    df['u_in_lag_back2'] = df.groupby('breath_id')['u_in'].shift(-2)
    df['u_out_lag_back2'] = df.groupby('breath_id')['u_out'].shift(-2)
    df['u_in_lag3'] = df.groupby('breath_id')['u_in'].shift(3)
    df['u_out_lag3'] = df.groupby('breath_id')['u_out'].shift(3)
    df['u_in_lag_back3'] = df.groupby('breath_id')['u_in'].shift(-3)
    df['u_out_lag_back3'] = df.groupby('breath_id')['u_out'].shift(-3)
    df['u_in_lag4'] = df.groupby('breath_id')['u_in'].shift(4)
    df['u_out_lag4'] = df.groupby('breath_id')['u_out'].shift(4)
    df['u_in_lag_back4'] = df.groupby('breath_id')['u_in'].shift(-4)
    df['u_out_lag_back4'] = df.groupby('breath_id')['u_out'].shift(-4)
    df = df.fillna(0)
    
    df['breath_id__u_in__max'] = df.groupby(['breath_id'])['u_in'].transform('max')
    df['breath_id__u_out__max'] = df.groupby(['breath_id'])['u_out'].transform('max')
    
    df['u_in_diff1'] = df['u_in'] - df['u_in_lag1']
    df['u_out_diff1'] = df['u_out'] - df['u_out_lag1']
    df['u_in_diff2'] = df['u_in'] - df['u_in_lag2']
    df['u_out_diff2'] = df['u_out'] - df['u_out_lag2']
    
    df['breath_id__u_in__diffmax'] = df.groupby(['breath_id'])['u_in'].transform('max') - df['u_in']
    df['breath_id__u_in__diffmean'] = df.groupby(['breath_id'])['u_in'].transform('mean') - df['u_in']
    
    df['breath_id__u_in__diffmax'] = df.groupby(['breath_id'])['u_in'].transform('max') - df['u_in']
    df['breath_id__u_in__diffmean'] = df.groupby(['breath_id'])['u_in'].transform('mean') - df['u_in']
    
    df['u_in_diff3'] = df['u_in'] - df['u_in_lag3']
    df['u_out_diff3'] = df['u_out'] - df['u_out_lag3']
    df['u_in_diff4'] = df['u_in'] - df['u_in_lag4']
    df['u_out_diff4'] = df['u_out'] - df['u_out_lag4']
    df['cross']= df['u_in']*df['u_out']
    df['cross2']= df['time_step']*df['u_out']
    
    df['R'] = df['R'].astype(str)
    df['C'] = df['C'].astype(str)
    df['R__C'] = df["R"].astype(str) + '__' + df["C"].astype(str)
    df = pd.get_dummies(df)
    return df

train = add_features(train)
test = add_features(test)

In [ ]:
train.head()

In [ ]:
train.shape

In [ ]:
targets = train[['pressure']].to_numpy().reshape(-1, 80)
train.drop(['pressure', 'id', 'breath_id'], axis=1, inplace=True)
test = test.drop(['id', 'breath_id'], axis=1)

In [ ]:
RS = RobustScaler()
train = RS.fit_transform(train)
test = RS.transform(test)

In [ ]:
train = train.reshape(-1, 80, train.shape[-1])
test = test.reshape(-1, 80, train.shape[-1])

In [ ]:
from tensorflow.keras.callbacks import Callback
import tensorflow.keras.backend as K
class WarmupExponentialDecay(Callback):
    def __init__(self,lr_base=0.0002,lr_min=0.0,decay=0,warmup_epochs=0):
        self.num_passed_batchs = 0   #?????
        self.warmup_epochs=warmup_epochs  
        self.lr=lr_base #learning_rate_base
        self.lr_min=lr_min #????????,???????
        self.decay=decay  #?????
        self.steps_per_epoch=0 #???????
        
    def on_batch_begin(self, batch, logs=None):
        # params????????Callback?????
        if self.steps_per_epoch==0:
            #?????????????
            if self.params['steps'] == None:
                self.steps_per_epoch = np.ceil(1. * self.params['samples'] / self.params['batch_size'])
            else:
                self.steps_per_epoch = self.params['steps']
        if self.num_passed_batchs < self.steps_per_epoch * self.warmup_epochs:
            K.set_value(self.model.optimizer.lr,
                        self.lr*(self.num_passed_batchs + 1) / self.steps_per_epoch / self.warmup_epochs)
        else:
            K.set_value(self.model.optimizer.lr,
                        self.lr*((1-self.decay)**(self.num_passed_batchs-self.steps_per_epoch*self.warmup_epochs)))
        self.num_passed_batchs += 1
        
    def on_epoch_begin(self,epoch,logs=None):
        #????????,????
        print("learning_rate:",K.get_value(self.model.optimizer.lr))

In [ ]:
EPOCH = 500
BATCH_SIZE = 1024
NUM_FOLDS = 5

# detect and init the TPU
tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()

# instantiate a distribution strategy
tpu_strategy = tf.distribute.experimental.TPUStrategy(tpu)

with tpu_strategy.scope():
    kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=402)
    test_preds = []
    for fold, (train_idx, test_idx) in enumerate(kf.split(train, targets)):
        print('-'*15, '>', f'Fold {fold+1}', '<', '-'*15)
        X_train, X_valid = train[train_idx], train[test_idx]
        y_train, y_valid = targets[train_idx], targets[test_idx]
        model = keras.models.Sequential([
            keras.layers.Input(shape=train.shape[-2:]),
            keras.layers.Bidirectional(keras.layers.LSTM(1024, return_sequences=True)),
            keras.layers.Bidirectional(keras.layers.LSTM(512, return_sequences=True)),
            keras.layers.Bidirectional(keras.layers.LSTM(256, return_sequences=True)),
            keras.layers.Bidirectional(keras.layers.LSTM(128, return_sequences=True)),
#             keras.layers.Bidirectional(keras.layers.LSTM(128, return_sequences=True)),
            keras.layers.Dense(128, activation='selu'),
#             keras.layers.Dropout(0.1),
            keras.layers.Dense(1),
        ])
        model.compile(optimizer="adam", loss="mae")

#         scheduler = ExponentialDecay(1e-3, 40*((len(train)*0.8)/BATCH_SIZE), 1e-5)
#         lr = LearningRateScheduler(scheduler, verbose=1)
        lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, verbose=1)
#         lr = WarmupExponentialDecay(lr_base=1e-3, decay=1e-5, warmup_epochs=30)
        es = EarlyStopping(monitor="val_loss", patience=60, verbose=1, mode="min", restore_best_weights=True)
    
        checkpoint_filepath = f"folds{fold}.hdf5"
        sv = keras.callbacks.ModelCheckpoint(
            checkpoint_filepath, monitor='val_loss', verbose=1, save_best_only=True,
            save_weights_only=False, mode='auto', save_freq='epoch',
            options=None
        )

        model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=EPOCH, batch_size=BATCH_SIZE, callbacks=[lr, es, sv])
        #model.save(f'Fold{fold+1} RNN Weights')
        test_preds.append(model.predict(test).squeeze().reshape(-1, 1).squeeze())

In [ ]:
!ls ./

In [ ]:
submission["pressure"] = sum(test_preds)/NUM_FOLDS
submission.to_csv('submission.csv', index=False)